# LangChain 한국어 입문 실습 노트북

이 노트북은 **LangChain을 처음 접하는 사람**이 OpenRouter를 통해 모델을 호출하고, 프롬프트 체인과 간단한 RAG까지 한 번에 경험할 수 있도록 구성한 실습 자료입니다.

## 이 노트북에서 배우는 것

- LangChain에서 모델을 연결하는 기본 패턴
- `ChatPromptTemplate | 모델 | StrOutputParser()` 형태의 가장 기본적인 체인
- 여러 요청을 한꺼번에 처리하는 `batch()` 사용법
- JSON 형태로 결과를 받아 후처리하는 방법
- 로컬 문서를 바탕으로 간단한 RAG를 만드는 방법

## 모델 사용 기준

- small 실습 모델: `OPENROUTER_SMALL_MODEL`
- large 실습 모델: `OPENROUTER_LARGE_MODEL`
- RAG 임베딩 모델: `OPENROUTER_EMBED_MODEL`

임베딩 모델은 벡터 검색을 위해 별도로 필요하므로, 채팅 모델과는 분리해서 사용합니다. 기본 `.env-example`은 멀티모달 임베딩 모델을 텍스트 문서 임베딩 용도로 사용하도록 구성되어 있습니다.

## 참고한 공식 자료

아래 공식 문서와 튜토리얼의 패턴을 초보자용으로 단순화해 실습 흐름으로 재구성했습니다.

- LangChain `ChatOpenAI` 통합 문서: https://docs.langchain.com/oss/python/integrations/chat/openai
- LangChain RAG 튜토리얼: https://python.langchain.com/docs/tutorials/rag/
- LangChain Structured Output 가이드: https://python.langchain.com/docs/how_to/structured_output/
- LangChain Text Splitter 개념 문서: https://python.langchain.com/docs/concepts/text_splitters/
- OpenRouter Quickstart: https://openrouter.ai/docs/quickstart
- OpenRouter API 개요: https://openrouter.ai/docs/api/reference/overview
- OpenRouter Embeddings 문서: https://openrouter.ai/docs/sdks/python/api-reference/embeddings

실전에서는 여기에 LangSmith 추적, 평가, 외부 벡터 DB를 추가하면 더 좋은 구조로 확장할 수 있습니다.

## 0. 설치와 실행 준비

먼저 프로젝트 루트에서 아래 패키지를 설치하세요.

```bash
pip install -r requirements.txt
```

필수 환경 변수:

- `OPENROUTER_API_KEY`

선택 환경 변수:

- `OPENROUTER_BASE_URL`
- `OPENROUTER_SMALL_MODEL`
- `OPENROUTER_LARGE_MODEL`
- `OPENROUTER_EMBED_MODEL`
- `OPENROUTER_SITE_URL` (선택)
- `OPENROUTER_APP_NAME` (선택)

이 노트북은 환경 설정 코드를 `src/langchain_cookbook/openrouter_setup.py`로 분리해 두었습니다.
해당 모듈은 프로젝트 루트의 `.env-example`을 먼저 읽고, `.env`가 있으면 그 값으로 덮어씁니다.
temperature, max_retries, tiktoken_enabled 같은 실행 옵션은 환경 변수가 아니라 코드에서 관리합니다.
따라서 먼저 해당 파일에 `OPENROUTER_API_KEY` 값을 넣어 주세요.

> 주의: OpenRouter의 무료 임베딩 엔드포인트는 입력이 로깅될 수 있으므로, 민감한 데이터는 사용하지 않는 편이 안전합니다.

In [1]:
import json
import re
import sys
import textwrap
from pathlib import Path

from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter

project_root = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "src" / "langchain_cookbook" / "openrouter_setup.py").exists():
        project_root = candidate
        break

if project_root is None:
    raise FileNotFoundError("src/langchain_cookbook/openrouter_setup.py 파일을 찾지 못했습니다.")

src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from langchain_cookbook.openrouter_setup import (
    load_openrouter_settings,
    make_large_chat_model,
    make_embeddings,
    make_small_chat_model,
)

settings = load_openrouter_settings()
OPENROUTER_BASE_URL = settings.base_url
SMALL_MODEL = settings.small_model
LARGE_MODEL = settings.large_model
EMBEDDING_MODEL = settings.embedding_model


None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [2]:
slm = make_small_chat_model()
llm = make_large_chat_model()
embeddings = make_embeddings()

print(f"slm 모델: {SMALL_MODEL}")
print(f"llm 모델: {LARGE_MODEL}")
print(f"임베딩 모델: {EMBEDDING_MODEL}")


slm 모델: nvidia/nemotron-3-nano-30b-a3b:free
llm 모델: nvidia/nemotron-3-super-120b-a12b:free
임베딩 모델: nvidia/llama-nemotron-embed-vl-1b-v2:free


## 1. 가장 단순한 호출부터 시작하기

LangChain을 처음 배울 때는 바로 복잡한 체인부터 만들기보다, **모델 하나를 안정적으로 호출하는 것**부터 익히는 편이 좋습니다.

아래 코드는 `ChatOpenAI` 객체를 통해 OpenRouter로 요청을 보내고, 결과의 `content`만 확인하는 가장 단순한 예제입니다.

In [3]:
first_response = slm.invoke(
    "LangChain이 무엇인지 한국어로 3문장만 사용해서 아주 쉽게 설명해줘."
)

print(first_response.content)

LangChain은 LLM(대규모 언어 모델)을 활용해 복잡한 작업을 쉽게 수행할 수 있게 해주는 프레임워크입니다.  
다양한 도구와 연결해 질문에 답하거나, 데이터를 검색하고, 여러 단계의 작업을 자동화할 수 있습니다.  
코딩 없이도 간단한 명령으로 AI와 대화형 애플리케이션을 만들 수 있게 해줍니다.


## 2. 초보자가 가장 먼저 익혀야 하는 체인 패턴

LangChain의 핵심 감각은 아래 한 줄에 들어 있습니다.

```python
prompt | model | parser
```

- `prompt`: 입력을 일정한 형식으로 구성
- `model`: 실제 LLM 호출
- `parser`: 응답을 우리가 쓰기 쉬운 형태로 변환

아래 실습에서는 `ChatPromptTemplate`으로 입력을 분리하고, `StrOutputParser()`로 순수 문자열만 받습니다.

In [4]:
study_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "당신은 랭체인 입문 튜터입니다. 설명은 쉽고 짧게 하고, 마지막 줄에 한 줄 실습 팁을 추가하세요.",
        ),
        (
            "human",
            "주제: {topic}\n학습자 수준: {level}\n위 조건에 맞는 설명을 작성해줘.",
        ),
    ]
)

study_chain = study_prompt | slm | StrOutputParser()

study_result = study_chain.invoke(
    {"topic": "PromptTemplate와 Output Parser", "level": "LangChain을 처음 보는 사람"}
)

print(study_result)


**PromptTemplate**  
- 프롬프트를 템플릿 형태로 미리 정의하고, 사용자 입력이나 변수값을 넣어 동적으로 만들 수 있게 해줍니다.  - `PromptTemplate.from_template("당신은 {role}입니다. {question}에 답해줘.")`처럼 작성하면, `{role}`와 `{question}` 자리에 실제 값을 넣어줄 수 있어요.

**Output Parser**  - LLM이 만든 텍스트를 구조화된 형태(예: 리스트, 딕셔너리)로 변환해주는 역할을 합니다.  
- `OutputParser` 계열 클래스(예: `StringOutputParser`, `JsonOutputParser`)를 사용하면 LLM이 반환한 원시 문자열을 원하는 데이터 구조로 바로 파싱할 수 있습니다.

**실습 팁**  
- 간단한 프롬프트와 파서 조합을 직접 만들어 보고, LLM이 반환한 텍스트가 원하는 형식으로 바뀌는지 확인해 보세요. 🚀


## 3. 여러 요청을 한꺼번에 처리하기

실무에서는 질문이 하나만 들어오지 않습니다. `batch()`를 사용하면 같은 체인을 여러 입력에 반복 적용할 수 있어 분류, 요약, 초안 생성 같은 작업에 편리합니다.

아래 예제는 사용자 질문을 보고 어떤 LangChain 주제로 연결하면 좋을지 분류합니다.

In [5]:
classification_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "당신은 랭체인 학습 코치입니다. 답변 형식은 반드시 '카테고리 | 이유 | 추천 다음 단계' 한 줄로만 작성하세요.",
        ),
        ("human", "질문: {question}"),
    ]
)

classification_chain = classification_prompt | slm | StrOutputParser()

questions = [
    "프롬프트 템플릿은 왜 필요한가요?",
    "여러 문서를 참고해서 답하게 만들고 싶어요.",
    "모델 응답을 JSON처럼 다루고 싶어요.",
]

classification_results = classification_chain.batch(
    [{"question": question} for question in questions]
)

for question, result in zip(questions, classification_results):
    print(f"질문: {question}")
    print(f"결과: {result}")
    print("-" * 80)


질문: 프롬프트 템플릿은 왜 필요한가요?
결과: 프롬프트 엔지니어링 | 템플릿을 사용하면 일관된 지시와 최적화된 출력을 보장해 효율성을 높입니다. | 현재 과제에 맞는 템플릿을 직접 작성해 보기
--------------------------------------------------------------------------------
질문: 여러 문서를 참고해서 답하게 만들고 싶어요.
결과: 문서 기반 답변 | 여러 문서를 종합하여 신뢰성 있는 답변을 생성하기 위해 필요 | RAG 시스템 구축 및 프롬프트 최적화
--------------------------------------------------------------------------------
질문: 모델 응답을 JSON처럼 다루고 싶어요.
결과: 요청 | 사용자가 모델 응답을 JSON 형태로 다루고 싶어 하는 경우, 구조화된 데이터 처리를 위한 프롬프트 설계가 필요함 | JSON 포맷 프롬프트를 사용하도록 가이드하고 실제 예시를 제공
--------------------------------------------------------------------------------


## 4. JSON 형태 결과 받기

공식 문서에서는 `with_structured_output()` 같은 구조화 출력 기능을 권장합니다. 다만 OpenRouter는 여러 제공자를 OpenAI 호환 API 뒤에 라우팅하기 때문에, **모델에게 JSON만 출력하라고 요구하고 LangChain에서 후처리하는 방식**이 더 안정적일 때가 많습니다.

아래 예제는 `.env-example`의 `OPENROUTER_LARGE_MODEL`에 지정된 large 모델을 사용해 학습 질문을 분석하고 JSON으로 정리합니다.

In [6]:
def extract_json_block(text: str) -> dict:
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if match is None:
        raise ValueError(f"JSON 객체를 찾지 못했습니다. 원문 응답: {text}")
    return json.loads(match.group())


analysis_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "당신은 LangChain 학습 질문 분석기입니다. 반드시 JSON 객체 하나만 반환하세요. 추가 설명은 금지합니다.",
        ),
        (
            "human",
            "질문: {question}\n\n"
            "아래 스키마를 따르세요.\n"
            "{{\n"
            '  "difficulty": "easy | medium | hard",\n'
            '  "core_concepts": ["개념1", "개념2"],\n'
            '  "next_step": "바로 해볼 실습 1개"\n'
            "}}",
        ),
    ]
)

analysis_chain = (
    analysis_prompt
    | llm
    | StrOutputParser()
    | RunnableLambda(extract_json_block)
)

analysis_result = analysis_chain.invoke(
    {"question": "PromptTemplate와 StrOutputParser는 각각 왜 필요하고, 언제 같이 쓰나요?"}
)

analysis_result


{'difficulty': 'easy',
 'core_concepts': ['PromptTemplate', 'StrOutputParser'],
 'next_step': '간단한 프롬프트 템플릿을 만들어 변수 하나를 넣고, LLM에 전달한 후 StrOutputParser를 붙여 출력 문자열을 받아보는 체인을 작성해 보세요.'}

## 5. 로컬 문서로 RAG 준비하기

이제 조금 더 실전다운 흐름으로 넘어가 보겠습니다. RAG는 보통 아래 순서로 구성합니다.

1. 문서를 준비한다.
2. 문서를 잘게 나눈다.
3. 임베딩을 만든다.
4. 벡터 저장소에 넣는다.
5. 질문과 가까운 문서를 찾아서 답변 생성에 넣는다.

여기서는 외부 파일 대신, 랭체인 학습용 짧은 문서를 직접 만들어서 인덱싱합니다.

In [3]:
study_docs = [
    Document(
        page_content=textwrap.dedent(
            """
            LangChain은 모델 호출을 단순히 한 번 실행하는 데서 끝나지 않고,
            프롬프트 구성, 출력 파싱, 문서 검색, 툴 사용 같은 단계를 연결할 수 있게 해주는 프레임워크다.
            입문자는 먼저 prompt, model, parser를 분리하는 습관을 들이는 것이 좋다.
            이렇게 하면 프롬프트 수정, 모델 교체, 후처리 로직 변경이 쉬워진다.
            """
        ).strip(),
        metadata={"source": "intro_note.md", "topic": "기초"},
    ),
    Document(
        page_content=textwrap.dedent(
            """
            좋은 프롬프트는 시스템 지시와 사용자 입력을 분리한다.
            특히 반복 실행되는 체인에서는 ChatPromptTemplate를 사용해 변수 자리를 명확히 두는 편이 유지보수에 유리하다.
            출력이 문자열인지, JSON인지, 도구 호출인지에 따라 후처리 전략도 달라진다.
            초보자 실습에서는 StrOutputParser로 시작한 뒤, 필요할 때 구조화 출력으로 확장하면 좋다.
            """
        ).strip(),
        metadata={"source": "prompt_note.md", "topic": "프롬프트"},
    ),
    Document(
        page_content=textwrap.dedent(
            """
            RAG에서는 긴 문서를 작은 조각으로 나누는 과정이 중요하다.
            RecursiveCharacterTextSplitter는 일반적인 텍스트를 다룰 때 자주 쓰는 기본 선택지다.
            chunk overlap을 두면 문장이 경계에서 잘릴 때도 앞뒤 맥락을 조금 유지할 수 있다.
            검색된 문맥만 바탕으로 답하라고 지시하고, 모르면 모른다고 답하게 해야 환각을 줄일 수 있다.
            """
        ).strip(),
        metadata={"source": "rag_note.md", "topic": "RAG"},
    ),
    Document(
        page_content=textwrap.dedent(
            """
            실전 서비스에서는 체인을 한 번에 크게 만들기보다 작은 단계로 나누는 편이 디버깅에 좋다.
            예를 들어 검색 단계, 문맥 포맷팅 단계, 최종 답변 생성 단계를 분리하면 어떤 단계에서 품질이 떨어지는지 찾기 쉽다.
            가능하면 근거 문서의 source를 함께 보여주고, LangSmith 같은 추적 도구로 실행 로그를 남기는 것이 좋다.
            """
        ).strip(),
        metadata={"source": "production_note.md", "topic": "운영"},
    ),
]

splitter = RecursiveCharacterTextSplitter(chunk_size=180, chunk_overlap=40)
splits = splitter.split_documents(study_docs)

vector_store = InMemoryVectorStore(embeddings)
vector_store.add_documents(splits)

print(f"원본 문서 수: {len(study_docs)}")
print(f"분할된 청크 수: {len(splits)}")
print("첫 번째 청크 미리보기:")
print(splits[0].page_content)
print(splits[0].metadata)


원본 문서 수: 4
분할된 청크 수: 8
첫 번째 청크 미리보기:
LangChain은 모델 호출을 단순히 한 번 실행하는 데서 끝나지 않고,
프롬프트 구성, 출력 파싱, 문서 검색, 툴 사용 같은 단계를 연결할 수 있게 해주는 프레임워크다.
입문자는 먼저 prompt, model, parser를 분리하는 습관을 들이는 것이 좋다.
{'source': 'intro_note.md', 'topic': '기초'}


## 6. 검색 결과를 넣어 답변 생성하기

이제 검색기(`retriever`)와 답변 생성 체인을 연결합니다.

여기서 중요한 프롬프트 규칙은 두 가지입니다.

- **검색된 문맥 안에서만 답하기**
- **문맥에 없으면 모른다고 말하기**

이 규칙은 공식 RAG 튜토리얼에서도 매우 중요하게 다루는 기본 원칙입니다.

In [4]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})


def format_docs(docs):
    return "\n\n".join(
        f"[source: {doc.metadata['source']}]\n{doc.page_content}" for doc in docs
    )


rag_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "당신은 LangChain 튜터입니다. 반드시 제공된 문맥만 사용해서 답하세요. 문맥에 없으면 '모르겠습니다.'라고 답하세요. 마지막 줄에는 참고한 source 이름을 콤마로 정리하세요.",
        ),
        (
            "human",
            "질문: {question}\n\n참고 문맥:\n{context}",
        ),
    ]
)

rag_chain = (
    {
        "question": RunnablePassthrough(),
        "context": retriever | RunnableLambda(format_docs),
    }
    | rag_prompt
    | llm
    | StrOutputParser()
)

question = "LangChain에서 chunk overlap을 두는 이유와, RAG에서 왜 모르면 모른다고 답하게 해야 하는지 설명해줘."

retrieved_docs = retriever.invoke(question)
print("[검색된 문서]")
for idx, doc in enumerate(retrieved_docs, start=1):
    print(f"{idx}. source={doc.metadata['source']}, topic={doc.metadata['topic']}")
    print(doc.page_content)
    print("-" * 80)

print("\n[RAG 답변]")
print(rag_chain.invoke(question))


[검색된 문서]
1. source=rag_note.md, topic=RAG
RAG에서는 긴 문서를 작은 조각으로 나누는 과정이 중요하다.
RecursiveCharacterTextSplitter는 일반적인 텍스트를 다룰 때 자주 쓰는 기본 선택지다.
chunk overlap을 두면 문장이 경계에서 잘릴 때도 앞뒤 맥락을 조금 유지할 수 있다.
--------------------------------------------------------------------------------
2. source=intro_note.md, topic=기초
LangChain은 모델 호출을 단순히 한 번 실행하는 데서 끝나지 않고,
프롬프트 구성, 출력 파싱, 문서 검색, 툴 사용 같은 단계를 연결할 수 있게 해주는 프레임워크다.
입문자는 먼저 prompt, model, parser를 분리하는 습관을 들이는 것이 좋다.
--------------------------------------------------------------------------------
3. source=production_note.md, topic=운영
실전 서비스에서는 체인을 한 번에 크게 만들기보다 작은 단계로 나누는 편이 디버깅에 좋다.
예를 들어 검색 단계, 문맥 포맷팅 단계, 최종 답변 생성 단계를 분리하면 어떤 단계에서 품질이 떨어지는지 찾기 쉽다.
--------------------------------------------------------------------------------

[RAG 답변]
chunk overlap을두면 문서를 작은 조각으로 나눌 때 문장이 경계에서 잘려도 앞뒤 맥락을 조금 유지할 수 있어, 의미가 끊기지 않도록 도와줍니다. (rag_note.md)

RAG에서 모르면 모른다고 답해야 하는 이유에 대해서는 제공된 문맥에 언급되어 있지 않으므로 모르겠습니다.

참고한 source: rag_note.md


## 마무리

이제 아래 흐름을 직접 실행해 본 상태가 됩니다.

- OpenRouter를 LangChain `ChatOpenAI`와 연결하기
- `ChatPromptTemplate`와 `StrOutputParser`로 기본 체인 만들기
- `batch()`로 반복 작업 처리하기
- 모델 응답을 JSON으로 받아 후처리하기
- 임베딩 + 벡터 저장소 + 검색 + 답변 생성으로 간단한 RAG 만들기

다음 단계로는 아래 확장을 추천합니다.

1. 지금 만든 `study_docs`를 실제 사내 문서나 개인 노트로 바꿔보기
2. `InMemoryVectorStore` 대신 Chroma, PGVector 같은 외부 저장소로 교체하기
3. LangSmith 추적을 붙여 어느 단계에서 품질이 흔들리는지 확인하기
4. 제공자 지원 상태가 맞는 경우 `with_structured_output()`과 tool calling도 실습해보기
